# Decision Tree Baseline

A second baseline that uses a tree-based model and all available features except identifiers and leakage columns. The dataset uses `repo_file` rather than `repo_link`, so that field is excluded too.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [5]:
base_dir = next(
    candidate for candidate in [
        Path.cwd() / "training" / "data",
        Path.cwd().parent / "data",
    ]
    if (candidate / "train_set.csv").exists()
)

train_df = pd.read_csv(base_dir / "train_set.csv")
val_df = pd.read_csv(base_dir / "val_set.csv")
test_df = pd.read_csv(base_dir / "test_set.csv")

target_col = "average_ms"
excluded_cols = {"model", "run_id", "repo_file", "repo_link", target_col, "stddev_ms", "min_ms", "max_ms"}

y_train = np.log(train_df[target_col].clip(lower=1e-9))
y_val = np.log(val_df[target_col].clip(lower=1e-9))
y_test = np.log(test_df[target_col].clip(lower=1e-9))

sparsity_thresholds = [0.99, 0.95, 0.75]

In [6]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,
        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),
        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,
        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }

def evaluate(model, split_name, features, target_log):
    prediction_log = model.predict(features)
    metrics = latency_metrics_from_log(target_log, prediction_log)
    print(f"{split_name}:")
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")

for sparsity_threshold in sparsity_thresholds:
    all_feature_cols = [col for col in train_df.columns if col not in excluded_cols]
    train_features = train_df[all_feature_cols]

    sparsity = train_features.apply(
        lambda col: ((col == 0) | col.isna()).mean()
        if pd.api.types.is_numeric_dtype(col)
        else col.isna().mean()
    )

    feature_cols = sparsity[sparsity <= sparsity_threshold].index.tolist()

    X_train = train_df[feature_cols]
    X_val = val_df[feature_cols]
    X_test = test_df[feature_cols]

    numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = [col for col in feature_cols if col not in numeric_features]

    preprocess = ColumnTransformer(
        [
            (
                "num",
                SimpleImputer(strategy="median"),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    model = Pipeline(
        [
            ("preprocess", preprocess),
            ("regressor", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2)),
        ]
    )

    _ = model.fit(X_train, y_train)

    print(f"\n=== Sparsity cutoff: {int(sparsity_threshold * 100)}% ===")
    print(f"kept_features: {len(feature_cols)} / {len(all_feature_cols)}")
    evaluate(model, "train", X_train, y_train)
    evaluate(model, "validation", X_val, y_val)
    evaluate(model, "test", X_test, y_test)


=== Sparsity cutoff: 99% ===
kept_features: 95 / 147
train:
  rmse_log: 0.1920
  rmse_ms: 133.1811
  rmse_percent: 24.0671
  median_relative_error: 0.0209
  p90_relative_error: 0.2096
  p95_relative_error: 0.3809
  median_percent_error: 2.0918
  p90_percent_error: 20.9584
  p95_percent_error: 38.0941
  median_ratio_error: 1.0211
  p90_ratio_error: 1.2349
  within_10pct: 0.7971
  within_25pct: 0.9176
  within_50pct: 0.9682
  within_2x: 0.9836
validation:
  rmse_log: 0.2179
  rmse_ms: 137.5748
  rmse_percent: 29.6880
  median_relative_error: 0.0336
  p90_relative_error: 0.2582
  p95_relative_error: 0.4475
  median_percent_error: 3.3601
  p90_percent_error: 25.8169
  p95_percent_error: 44.7538
  median_ratio_error: 1.0341
  p90_ratio_error: 1.2921
  within_10pct: 0.7327
  within_25pct: 0.8960
  within_50pct: 0.9589
  within_2x: 0.9780
test:
  rmse_log: 0.2099
  rmse_ms: 109.6809
  rmse_percent: 24.5539
  median_relative_error: 0.0326
  p90_relative_error: 0.2623
  p95_relative_error: 0.4

KeyboardInterrupt: 

test:
  rmse_log: 0.2099
  rmse_ms: 109.6320
  rmse_percent: 24.5656
  median_relative_error: 0.0326
  p90_relative_error: 0.2623
  p95_relative_error: 0.4492
  median_percent_error: 3.2606
  p90_percent_error: 26.2284
  p95_percent_error: 44.9185
  median_ratio_error: 1.0331
  p90_ratio_error: 1.2982
  within_10pct: 0.7349
  within_25pct: 0.8941
  within_50pct: 0.9590
  within_2x: 0.9784